<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/verifiers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 117.8 MB/s eta 0:00:00


In [ ]:
import torch
import transformers

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)

Torch: 2.11.0+cu128
CUDA: 12.8
GPU available: True
Transformers: 5.15.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
question = "What is 18 multiplied by 7?"

ground_truth = 126

In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a math assistant. "
            "Solve the problem and finish your response exactly with "
            "'Final answer: <number>'."
        )
    },
    {
        "role": "user",
        "content": question
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(prompt)

<|im_start|>system
You are a math assistant. Solve the problem and finish your response exactly with 'Final answer: <number>'.<|im_end|>
<|im_start|>user
What is 18 multiplied by 7?<|im_end|>
<|im_start|>assistant



In [ ]:
import re
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,

        max_new_tokens=80,

        # sampling gives us different candidates
        do_sample=True,
        temperature=0.9,
        top_p=0.95,

        # generate multiple candidate answers
        num_return_sequences=5,
    )

In [ ]:
candidates = []

input_length = inputs["input_ids"].shape[1]

for output in outputs:

    generated_tokens = output[input_length:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    candidates.append(text)

In [ ]:
for i, candidate in enumerate(candidates, start=1):

    print(f"\n--- Candidate {i} ---")
    print(candidate)


--- Candidate 1 ---
Final answer: 126.

--- Candidate 2 ---
18 multiplied by 7 equals 126. Final answer: 126.

--- Candidate 3 ---
Final answer: 126.

--- Candidate 4 ---
Final answer: 126.

--- Candidate 5 ---
18 multiplied by 7 is 126. Therefore, the final answer is 126.


In [ ]:
def extract_final_answer(text):

    match = re.search(
        r"Final answer:\s*(-?\d+(?:\.\d+)?)",
        text,
        re.IGNORECASE
    )

    if match is None:
        return None

    return float(match.group(1))

In [ ]:
extract_final_answer(
    "18 times 7 is 126. Final answer: 126"
)

126.0

In [ ]:
def verifier(candidate, ground_truth):

    predicted = extract_final_answer(candidate)

    if predicted is None:
        return 0

    if predicted == float(ground_truth):
        return 1

    return 0

In [ ]:
scores = []

for i, candidate in enumerate(candidates, start=1):

    predicted = extract_final_answer(candidate)

    score = verifier(
        candidate,
        ground_truth
    )

    scores.append(score)

    print(
        f"Candidate {i}: "
        f"predicted={predicted}, "
        f"reward={score}"
    )

Candidate 1: predicted=126.0, reward=1
Candidate 2: predicted=126.0, reward=1
Candidate 3: predicted=126.0, reward=1
Candidate 4: predicted=126.0, reward=1
Candidate 5: predicted=None, reward=0


In [ ]:
best_index = scores.index(max(scores))

best_candidate = candidates[best_index]

print("Selected response:\n")
print(best_candidate)

print("\nVerifier score:", scores[best_index])

Selected response:

Final answer: 126.

Verifier score: 1


In [ ]:
questions = [
    {
        "question": "What is 18 multiplied by 7?",
        "ground_truth": 126
    },
    {
        "question": "What is 24 multiplied by 9?",
        "ground_truth": 216
    },
    {
        "question": "What is 144 divided by 12?",
        "ground_truth": 12
    },
    {
        "question": "What is 35 + 48?",
        "ground_truth": 83
    },
    {
        "question": "What is 100 - 37?",
        "ground_truth": 63
    }
]

In [ ]:
def generated_candidates(question, num_candidates=5):
  messages = [
      {
          "role": "system",
          "content": (
              "You are a math assistant."
              "Solve the problem and always finish with "
              "'Final answer: <number>'."
          )
      },
      {
          "role": "user",
          "content": question
      }
  ]

  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt"
  ).to(model.device)

  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        num_return_sequences=num_candidates,
    )

  input_length = inputs["input_ids"].shape[1]

  candidates = []

  for output in outputs:

    generated_tokens = output[input_length:]

    text=tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    candidates.append(text)

  return candidates


In [ ]:
import re

def extract_final_answer(text):

  match = re.search(
      r"Final answer:\s*(-?\d+(?:\.\d+)?)",
      text,
      re.IGNORECASE
  )

  if match is None:
    return None

  return float(match.group(1))

In [ ]:
results = []

for q_index, item in enumerate(questions, start=1):

  question = item["question"]
  ground_truth = item["ground_truth"]

  print("\n" + "=" * 60)
  print(f"Question {q_index}: {question}")
  print(f"Ground truth: {ground_truth}")
  print("=" * 60)

  candidates = generated_candidates(
      question,
      num_candidates=5

  )

  scores = []

  for i, candidate in enumerate(candidates, start=1):

    predicted = extract_final_answer(candidate)

    reward = verifier(
        candidate,
        ground_truth
    )

    scores.append(reward)


    print(
          f"Candidate {i}: "
          f"predicted={predicted}, "
          f"reward={reward}"
      )

  results.append({
          "question": question,
          "ground_truth": ground_truth,
          "candidates": candidates,
          "scores": scores
      })


Question 1: What is 18 multiplied by 7?
Ground truth: 126
Candidate 1: predicted=126.0, reward=1
Candidate 2: predicted=None, reward=0
Candidate 3: predicted=None, reward=0
Candidate 4: predicted=None, reward=0
Candidate 5: predicted=126.0, reward=1

Question 2: What is 24 multiplied by 9?
Ground truth: 216
Candidate 1: predicted=None, reward=0
Candidate 2: predicted=216.0, reward=1
Candidate 3: predicted=216.0, reward=1
Candidate 4: predicted=None, reward=0
Candidate 5: predicted=216.0, reward=1

Question 3: What is 144 divided by 12?
Ground truth: 12
Candidate 1: predicted=12.0, reward=1
Candidate 2: predicted=None, reward=0
Candidate 3: predicted=None, reward=0
Candidate 4: predicted=12.0, reward=1
Candidate 5: predicted=None, reward=0

Question 4: What is 35 + 48?
Ground truth: 83
Candidate 1: predicted=83.0, reward=1
Candidate 2: predicted=83.0, reward=1
Candidate 3: predicted=83.0, reward=1
Candidate 4: predicted=83.0, reward=1
Candidate 5: predicted=None, reward=0

Question 5: 